In [2]:
import time
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Download model (only once)
model_path = Path("face_landmarker.task")
if not model_path.exists():
    model_url = (
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
        "face_landmarker/float16/1/face_landmarker.task"
    )
    print("Downloading model...", model_url)
    urlretrieve(model_url, model_path)

# Setup model
options = vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(model_path)),
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

cap = cv2.VideoCapture(2)  # change if needed

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

with vision.FaceLandmarker.create_from_options(options) as landmarker:
    print("Press 'q' to quit")

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int(time.time() * 1000)

        result = landmarker.detect_for_video(mp_image, timestamp_ms)
        output = frame.copy()

        if result.face_landmarks:
            h, w = output.shape[:2]

            for face_landmarks in result.face_landmarks:
                points = []

                # Draw all face landmarks
                for lm in face_landmarks:
                    x = int(lm.x * w)
                    y = int(lm.y * h)
                    points.append((x, y))
                    cv2.circle(output, (x, y), 1, (0, 255, 255), -1)

                # Example: Detect smile (very basic heuristic 😄)
                left_mouth = points[61]
                right_mouth = points[291]

                if abs(left_mouth[0] - right_mouth[0]) > 60:
                    cv2.putText(output, "SMILE 😄", (10, 40),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

                # Example: Detect mouth open
                top_lip = points[13]
                bottom_lip = points[14]

                if abs(top_lip[1] - bottom_lip[1]) > 15:
                    cv2.putText(output, "MOUTH OPEN", (10, 80),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.imshow("Face Landmarks (MediaPipe Tasks)", output)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

Press 'q' to quit
